In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# %% [markdown]
# ## 1. Summary of Tables and Row Counts

# %%
with duckdb.connect(str(db_path), read_only=True) as conn:
    tables = conn.execute("SELECT table_name FROM duckdb_tables();").fetchall()

    data = []
    for (table_name,) in tables:
        count = conn.execute(f'SELECT count(*) FROM "{table_name}"').fetchone()[0]
        data.append({"table_name": table_name, "row_count": count})

    df_summary = (
        pd.DataFrame(data)
        .sort_values(by="row_count", ascending=False)
        .reset_index(drop=True)
    )

display(df_summary)

,table_name,row_count
0,openelectricity_mix,623962
1,aemo_nem_dispatch,533720
2,aemo_wem_dispatch,106594
3,bom_observations,53856
4,aemo_holidays,930


# Data Frequency

## BOM — Daily Frequency

- **Granularity:** 5 minutes
- **Readings per station per day:** 288
- **Number of stations:** 6
- **Total readings per day:** 1,728

### Calculation

- 24 hours × 60 minutes ÷ 5 minutes = **288 readings per station per day**
- 288 readings × 6 stations = **1,728 readings per day**

In [24]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# --- CONFIGURATION ---
TABLE_NAME = "bom_observations"
START_DATE = "2026-07-01"
END_DATE = "2026-08-06"
EXPECTED_DAILY_COUNT = 1728
# ---------------------

# %% [markdown]
# ## Completeness Check Query

# %%
query = f"""
SELECT
    cal.record_date,
    COUNT(d.ts) AS total_count
FROM (
    SELECT unnest(generate_series(
        '{START_DATE}'::timestamp,
        '{END_DATE}'::timestamp,
        '1 day'::interval
    ))::date AS record_date
) cal
LEFT JOIN {TABLE_NAME} d
       ON DATE(d.ts) = cal.record_date
      AND d.ts >= '{START_DATE} 00:00:00+00'
      AND d.ts < '{END_DATE} 00:00:00+00'
GROUP BY cal.record_date
HAVING COUNT(d.ts) < {EXPECTED_DAILY_COUNT}
ORDER BY cal.record_date ASC;
"""

with duckdb.connect(str(db_path), read_only=True) as conn:
    df_gaps = conn.execute(query).fetchdf()

display(
    HTML(
        f"<h3>⚠️ Incomplete Days Found (< {EXPECTED_DAILY_COUNT} records): {len(df_gaps)}</h3>"
    )
)
display(df_gaps)

,record_date,total_count
0,2026-07-01,108
1,2026-07-02,144
2,2026-07-03,144
3,2026-07-04,144
4,2026-07-05,144
5,2026-07-06,144
6,2026-07-07,144
7,2026-07-08,144
8,2026-07-09,144
9,2026-07-10,144


# Data Frequency

## OpenElectricity / OpenNEM (`ds-openelectricity`)

- **Regions:** 5 NEM regions
- **Primary granularity:** 30 minutes

| Granularity | Readings per Region / Day | × 5 Regions / Day | × 1 Year |
|---|---:|---:|---:|
| 5 minutes | 288 | 1,440 | 525,600 |
| 15 minutes | 96 | 480 | 175,200 |
| 30 minutes | 48 | 240 | 87,600 |
| Hourly | 24 | 120 | 43,800 |
| Daily | 1 | 5 | 1,825 |

### Calculation

- **5 minutes:** 288 × 5 × 365 = **525,600 readings/year**
- **15 minutes:** 96 × 5 × 365 = **175,200 readings/year**
- **30 minutes:** 48 × 5 × 365 = **87,600 readings/year**
- **Hourly:** 24 × 5 × 365 = **43,800 readings/year**
- **Daily:** 1 × 5 × 365 = **1,825 readings/year**

In [25]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# --- CONFIGURATION ---
TABLE_NAME = "openelectricity_mix"
START_DATE = "2025-08-01"
END_DATE = "2025-12-01"
EXPECTED_DAILY_COUNT = 1440
# ---------------------

# %% [markdown]
# ## Completeness Check Query

# %%
query = f"""
SELECT
    cal.record_date,
    COUNT(d.ts) AS total_count
FROM (
    SELECT unnest(generate_series(
        '{START_DATE}'::timestamp,
        '{END_DATE}'::timestamp,
        '1 day'::interval
    ))::date AS record_date
) cal
LEFT JOIN {TABLE_NAME} d
       ON DATE(d.ts) = cal.record_date
      AND d.ts >= '{START_DATE} 00:00:00+00'
      AND d.ts < '{END_DATE} 00:00:00+00'
GROUP BY cal.record_date
HAVING COUNT(d.ts) < {EXPECTED_DAILY_COUNT}
ORDER BY cal.record_date ASC;
"""

with duckdb.connect(str(db_path), read_only=True) as conn:
    df_gaps = conn.execute(query).fetchdf()

display(
    HTML(
        f"<h3>⚠️ Incomplete Days Found (< {EXPECTED_DAILY_COUNT} records): {len(df_gaps)}</h3>"
    )
)
display(df_gaps)

,record_date,total_count
0,2025-08-01,600
1,2025-10-31,1128
2,2025-11-01,600
3,2025-12-01,432


# Data Frequency

## AEMO NEM (`ds-aemo-nem`)

- **Regions:** 5 NEM regions
- **Primary granularity:** 5 minutes

| Granularity | Readings per Region / Day | × 5 Regions / Day | × 1 Year |
|---|---:|---:|---:|
| 5 minutes | 288 | 1,440 | 525,600 |
| 15 minutes | 96 | 480 | 175,200 |
| 30 minutes | 48 | 240 | 87,600 |
| Hourly | 24 | 120 | 43,800 |
| Daily | 1 | 5 | 1,825 |

### Calculation

- **5 minutes:** 288 × 5 × 365 = **525,600 readings/year**
- **15 minutes:** 96 × 5 × 365 = **175,200 readings/year**
- **30 minutes:** 48 × 5 × 365 = **87,600 readings/year**
- **Hourly:** 24 × 5 × 365 = **43,800 readings/year**
- **Daily:** 1 × 5 × 365 = **1,825 readings/year**

In [26]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# --- CONFIGURATION ---
TABLE_NAME = "aemo_nem_dispatch"
START_DATE = "2025-08-01"
END_DATE = "2026-08-04"
EXPECTED_DAILY_COUNT = 1440
# ---------------------

# %% [markdown]
# ## Completeness Check Query

# %%
query = f"""
SELECT
    cal.record_date,
    COUNT(d.ts) AS total_count
FROM (
    SELECT unnest(generate_series(
        '{START_DATE}'::timestamp,
        '{END_DATE}'::timestamp,
        '1 day'::interval
    ))::date AS record_date
) cal
LEFT JOIN {TABLE_NAME} d
       ON DATE(d.ts) = cal.record_date
      AND d.ts >= '{START_DATE} 00:00:00+00'
      AND d.ts < '{END_DATE} 00:00:00+00'
GROUP BY cal.record_date
HAVING COUNT(d.ts) < {EXPECTED_DAILY_COUNT}
ORDER BY cal.record_date ASC;
"""

with duckdb.connect(str(db_path), read_only=True) as conn:
    df_gaps = conn.execute(query).fetchdf()

display(
    HTML(
        f"<h3>⚠️ Incomplete Days Found (< {EXPECTED_DAILY_COUNT} records): {len(df_gaps)}</h3>"
    )
)
display(df_gaps)

,record_date,total_count
0,2025-08-01,1080
1,2026-03-10,1335
2,2026-08-04,360


# Data Frequency

## AEMO WEM (`ds-aemo-wem`)

- **Region:** 1 WEM region
- **Primary granularity:** 30 minutes

| Granularity | Readings per Region / Day | × 1 Region / Day | × 1 Year |
|---|---:|---:|---:|
| 5 minutes | 288 | 288 | 105,120 |
| 15 minutes | 96 | 96 | 35,040 |
| 30 minutes | 48 | 48 | 17,520 |
| Hourly | 24 | 24 | 8,760 |
| Daily | 1 | 1 | 365 |

### Calculation

- **5 minutes:** 288 × 1 × 365 = **105,120 readings/year**
- **15 minutes:** 96 × 1 × 365 = **35,040 readings/year**
- **30 minutes:** 48 × 1 × 365 = **17,520 readings/year**
- **Hourly:** 24 × 1 × 365 = **8,760 readings/year**
- **Daily:** 1 × 1 × 365 = **365 readings/year**

In [27]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# --- CONFIGURATION ---
TABLE_NAME = "aemo_wem_dispatch"
START_DATE = "2025-08-01"
END_DATE = "2026-08-04"
EXPECTED_DAILY_COUNT = 288
# ---------------------

# %% [markdown]
# ## Completeness Check Query

# %%
query = f"""
SELECT
    cal.record_date,
    COUNT(d.ts) AS total_count
FROM (
    SELECT unnest(generate_series(
        '{START_DATE}'::timestamp,
        '{END_DATE}'::timestamp,
        '1 day'::interval
    ))::date AS record_date
) cal
LEFT JOIN {TABLE_NAME} d
       ON DATE(d.ts) = cal.record_date
      AND d.ts >= '{START_DATE} 00:00:00+00'
      AND d.ts < '{END_DATE} 00:00:00+00'
GROUP BY cal.record_date
HAVING COUNT(d.ts) < {EXPECTED_DAILY_COUNT}
ORDER BY cal.record_date ASC;
"""

with duckdb.connect(str(db_path), read_only=True) as conn:
    df_gaps = conn.execute(query).fetchdf()

display(
    HTML(
        f"<h3>⚠️ Incomplete Days Found (< {EXPECTED_DAILY_COUNT} records): {len(df_gaps)}</h3>"
    )
)
display(df_gaps)

,record_date,total_count
0,2025-08-01,216
1,2026-08-04,72
